In [ ]:
pip install opencv-python

In [ ]:
pip install mediapipe

In [ ]:
pip install pyserial

In [ ]:
import serial 
import time 

ser = serial.Serial('/dev/ttyUSB0',baudrate=115200,bytesize =8, parity ='N', stopbits =1)
hex = '3A0100020003000400'
#hex = '3A0101020103010401'

data_send = bytes.fromhex(hex)
ser.write(data_send)

ser.close()

# Original code

In [ ]:
import cv2
import mediapipe as mp
import threading
import serial
import time
import numpy as np  # For smoothing detection with history buffer
import signal  # For handling Ctrl+C interruptions

# Initialize serial communication with error handling
try:
    ser = serial.Serial('/dev/ttyUSB0', 115200, timeout=1)  # Add timeout for safety
except serial.SerialException as e:
    print(f"Error: {e}")
    exit(1)

# Initialize webcam with error check
webcam = cv2.VideoCapture(2)
if not webcam.isOpened():
    print("Error: Could not open webcam.")
    exit(1)

# Initialize Mediapipe hands module
mpHands = mp.solutions.hands
hands = mpHands.Hands(static_image_mode=False, max_num_hands=1,
                      min_detection_confidence=0.8, min_tracking_confidence=0.7)
mpDraw = mp.solutions.drawing_utils

# Global variables for state tracking
last_finger_count = -1  # Store last detected finger count
finger_history = []  # Buffer to store recent counts for smoothing
cooldown_lock = threading.Lock()  # Lock to prevent race conditions
motor_running = False  # Track whether the motor is currently running

# ========== Helper Functions ==========
def stop_motor():
    """Sends the stop command to the motor."""
    try:
        ser.write(bytes.fromhex('3A01000200'))  # Stop command (Hex)
        print("Motor stopped safely.")
    except serial.SerialException as e:
        print(f"Error sending stop command: {e}")

def send_command(command, duration):
    """Send a motor command and hold it for 'duration' seconds."""
    global motor_running
    motor_running = True  # Mark motor as running

    with cooldown_lock:
        try:
            print(f"Sending command: {command}, holding for {duration} seconds.")
            ser.write(bytes.fromhex(command))  # Send command to motor
            time.sleep(duration)  # Hold for the specified duration
            stop_motor()  # Stop motor after the command
        except serial.SerialException as e:
            print(f"Error sending motor command: {e}")

    motor_running = False  # Mark motor as stopped

def smooth_finger_count(current_count, max_history=5):
    """Smooth the finger count by taking the most frequent (mode) count."""
    finger_history.append(current_count)
    if len(finger_history) > max_history:
        finger_history.pop(0)  # Keep only the last 'max_history' counts

    # Return the most common count from the history buffer (mode)
    return int(np.bincount(finger_history).argmax())

def signal_handler(sig, frame):
    """Handle Ctrl+C interruption and release resources safely."""
    print("Program interrupted! Stopping motor and releasing resources...")
    stop_motor()
    cleanup()
    exit(0)

def cleanup():
    """Release resources gracefully."""
    print("Releasing resources...")
    webcam.release()
    ser.close()
    cv2.destroyAllWindows()
    print("Resources released. Exiting.")

# Register the signal handler for safe termination on Ctrl+C
signal.signal(signal.SIGINT, signal_handler)

# ========== Helper Function for Hand Orientation ==========
def is_palm_facing(lm_list):
    wrist_y = lm_list[0][2]  # y-coordinate of wrist
    base_finger_y = lm_list[9][2]  # y-coordinate of the middle finger base
    return wrist_y < base_finger_y  # Palm facing if wrist is below the base of the fingers

# ========== Updated Finger Counting Logic ==========
def finger_counting(imageFrame):
    """Detect extended fingers and count them accurately."""
    global last_finger_count

    # Convert frame to RGB (Mediapipe requires RGB input)
    imgRGB = cv2.cvtColor(imageFrame, cv2.COLOR_BGR2RGB)

    # Process the frame with Mediapipe hands
    results = hands.process(imgRGB)

    # Initialize finger count
    finger_count = 0

    # If hands are detected
    if results.multi_hand_landmarks:
        for hand_landmark, hand_info in zip(results.multi_hand_landmarks, results.multi_handedness):
            # Get hand label (Left or Right)
            hand_label = hand_info.classification[0].label
            fingers = [False] * 5  # Initialize a list to track each finger
            lm_list = []

            # Extract landmark positions
            for id, lm in enumerate(hand_landmark.landmark):
                h, w, _ = imageFrame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                lm_list.append([id, cx, cy])


            # Determine hand orientation (palm facing or back of hand facing)
            palm_facing = is_palm_facing(lm_list)

            if palm_facing:
                # Palm facing: regular finger checking logic
                if hand_label == "Right":
                    # For the right hand, thumb tip should be on the right side of the IP joint
                    if lm_list[4][1] > lm_list[3][1]:  # Thumb
                        fingers[0] = True
                else:  # Left hand
                    # For the left hand, thumb tip should be on the left side of the IP joint
                    if lm_list[4][1] < lm_list[3][1]:  # Thumb
                        fingers[0] = True
            else:
                # Back of hand facing: reversed logic for thumb
                if hand_label == "Right":
                    # For the right hand, thumb tip should be on the left side of the IP joint
                    if lm_list[4][1] < lm_list[3][1]:  # Thumb
                        fingers[0] = True
                else:  # Left hand
                    # For the left hand, thumb tip should be on the right side of the IP joint
                    if lm_list[4][1] > lm_list[3][1]:  # Thumb
                        fingers[0] = True

            # Check each finger's state (extended or folded)
            if lm_list[8][2] < lm_list[7][2]:  # Index finger
                fingers[1] = True
            if lm_list[12][2] < lm_list[11][2]:  # Middle finger
                fingers[2] = True
            if lm_list[16][2] < lm_list[15][2]:  # Ring finger
                fingers[3] = True
            if lm_list[20][2] < lm_list[19][2]:  # Little finger
                fingers[4] = True

            # Count how many fingers are extended
            finger_count = sum(fingers)

            # Draw landmarks
            mpDraw.draw_landmarks(imageFrame, hand_landmark, mpHands.HAND_CONNECTIONS)

    # Display finger count
    cv2.putText(imageFrame, f"Finger Count: {finger_count}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 98), 3)

    # Smooth the count and send commands accordingly
    finger_count = smooth_finger_count(finger_count)

    if finger_count != last_finger_count:
        last_finger_count = finger_count

        # Define motor commands for each count
        command_map = {
            5: ('3A01010200', 5),
            4: ('3A01010200', 4),
            3: ('3A01010200', 3),
            2: ('3A01010200', 2),
            1: ('3A01010200', 1),
            0: ('3A01000200', 0)
        }

        if finger_count in command_map:
            command, duration = command_map[finger_count]
            threading.Thread(target=send_command, args=(command, duration)).start()

# ========== Main Program Loop ==========
try:
    while True:
        ret, imageFrame = webcam.read()
        if not ret:
            print("Error: Failed to read frame.")
            break

        finger_counting(imageFrame)
        cv2.imshow("Hand Gesture Finger Count", imageFrame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    stop_motor()
    cleanup()
